# Fine-tune YOLO segmentation on Nebius

Export a FiftyOne dataset to YOLO format, fine-tune **YOLO26-seg** in a [Nebius Serverless AI job](https://docs.nebius.com/serverless/jobs), pull the weights back, and run `apply_model`.

```
FiftyOne ──► YOLO export ──► Nebius bucket ──► Nebius GPU job ──► best.pt ──► apply_model
```

**Before you start:** follow the [README](README.md#setup) once to set up `uv`, the Nebius CLI, a bucket, and an S3 key profile named `nebius`. Then set `BUCKET` in the next cell.

In [ ]:
import json
import shlex
import shutil
import subprocess
import tarfile
import time
from pathlib import Path

import boto3
import fiftyone as fo
import fiftyone.types as fot
import fiftyone.utils.random as four
from fiftyone import ViewField as F
from fiftyone.utils.huggingface import load_from_hub
from ultralytics import YOLO

BUCKET = "your-bucket"  # the bucket you created
REGION = "eu-north1"  # region of your bucket and project
AWS_PROFILE = "nebius"  # the AWS CLI profile holding your Nebius S3 keys
PLATFORM = "gpu-l40s-a"  # 1x L40S; `nebius compute platform list` shows what your project has
PRESET = "1gpu-8vcpu-32gb"
TIMEOUT = "1h"  # billing cap; the minimum Nebius allows
IMAGE = "ultralytics/ultralytics:8.4.37"  # official image, nothing to build
MODEL = "yolo26s-seg.pt"
EPOCHS = 30
RUN_NAME = "plantseg-seg-v1"  # job name and output folder; change it to launch a new job
LABEL_FIELD = "segmentations"  # the Polylines field that holds your polygons

WORK_DIR = Path("nebius_run")
EXPORT_DIR = WORK_DIR / "plantseg-yolo"
ARCHIVE = WORK_DIR / "plantseg-yolo.tar.gz"
WEIGHTS = WORK_DIR / "best.pt"

## 1. Load the data

[PlantSeg-Test](https://huggingface.co/datasets/Voxel51/PlantSeg-Test): 1,200 leaf photos with polygon masks for plant diseases.

In [ ]:
dataset = load_from_hub("Voxel51/PlantSeg-Test", name="plantseg-test", overwrite=True)
session = fo.launch_app(dataset)

**Using your own data?** If you exported a dataset from the Voxel51 platform (or anywhere) in FiftyOne dataset format, load it from disk instead of the Hub. Point `dataset_dir` at the export folder, and set `LABEL_FIELD` in the config cell to your polygon field (it must be a `Polylines` field):

```python
dataset = fo.Dataset.from_dir(
    dataset_dir="/path/to/export",
    dataset_type=fot.FiftyOneDataset,
    name="my-dataset",
)
```

## 2. Export in YOLO segmentation format

Polygons are in `segmentations`. YOLO needs at least 3 points per polygon, so the first line drops the one that has fewer. Then split 80/20 and export.

In [ ]:
view = dataset.filter_labels(LABEL_FIELD, F("points")[0].length() >= 3)

# Clear old tags so re-running the cell doesn't leak samples between splits
view.untag_samples(["train", "val"])
four.random_split(view, {"train": 0.8, "val": 0.2}, seed=51)
classes = sorted(view.distinct(f"{LABEL_FIELD}.polylines.label"))

shutil.rmtree(EXPORT_DIR, ignore_errors=True)
for split in ("train", "val"):
    view.match_tags(split).export(
        export_dir=str(EXPORT_DIR),
        dataset_type=fot.YOLOv5Dataset,
        label_field=LABEL_FIELD,
        classes=classes,
        split=split,
        progress=False,
    )
print(view.count_sample_tags())

## 3. Upload to your Nebius bucket

One archive instead of thousands of small files: bucket mounts are slow for those, so the job unpacks it onto local disk.

In [ ]:
with tarfile.open(ARCHIVE, "w:gz") as tar:
    tar.add(EXPORT_DIR, arcname=EXPORT_DIR.name)

# Nebius Object Storage is S3-compatible; boto3 reads the keys from the AWS profile
session = boto3.Session(profile_name=AWS_PROFILE)
s3 = session.client("s3", endpoint_url=f"https://storage.{REGION}.nebius.cloud", region_name=REGION)
s3.upload_file(str(ARCHIVE), BUCKET, "datasets/plantseg-yolo.tar.gz")

## 4. Launch the job

[`train_seg_job.py`](train_seg_job.py) unpacks the data, trains, and writes the weights back to the bucket. The job runs it inside the stock Ultralytics image, so there is nothing to build.

The dry-run validates the command without creating anything. The next cell **starts a billable job**, capped at `TIMEOUT`. See [nebius.com/prices](https://nebius.com/prices).

In [ ]:
PROJECT_ID = subprocess.check_output(["nebius", "config", "get", "parent-id"], text=True).strip()

JOB_CMD = shlex.join([
    "ai", "job", "create", "--parent-id", PROJECT_ID, "--name", RUN_NAME,
    "--image", IMAGE, "--platform", PLATFORM, "--preset", PRESET, "--timeout", TIMEOUT,
    "--volume", f"s3://{BUCKET}:/bucket:rw:{AWS_PROFILE}",  # bucket mounted at /bucket
    "--inject-file", "train_seg_job.py:/app/train_seg_job.py",  # ships the script into the container
    "--container-command", "python3",
    "--args", f"/app/train_seg_job.py --model {MODEL} --epochs {EPOCHS} --run-name {RUN_NAME}",
])
!nebius {JOB_CMD} --dry-run

In [ ]:
!nebius {JOB_CMD} --async

## 5. Wait for it

Polls every 30 seconds, then prints the end of the log. Add `--tail 100` to the logs command to see more.

In [ ]:
get_job = ["nebius", "ai", "job", "get-by-name", "--name", RUN_NAME, "--parent-id", PROJECT_ID, "--format", "json"]

state = None
while True:
    job = json.loads(subprocess.check_output(get_job, text=True))
    if job["status"]["state"] != state:
        state = job["status"]["state"]
        print(state)
    if state in {"COMPLETED", "FAILED", "CANCELLED", "ERROR"}:
        break
    time.sleep(30)

!nebius ai job logs {job["metadata"]["id"]} --tail 20

## 6. Pull the weights and run `apply_model`

The job copied `best.pt` to `runs/<run-name>/weights/` in your bucket. Download it and run it on the validation split, the images it never trained on.

In [ ]:
s3.download_file(BUCKET, f"runs/{RUN_NAME}/weights/best.pt", str(WEIGHTS))

model = YOLO(str(WEIGHTS))
val_view = view.match_tags("val")
val_view.apply_model(model, label_field="predictions", confidence_thresh=0.25)
session.view = val_view

Predicted masks are in the `predictions` field. Compare them with `segmentations` in the App.

**Billing stops** when the job reaches a final state. To stop one early: `nebius ai job cancel --id <job-id>`.